# Qwen Fine-Tuning on ROCm — CFPB Complaint Categorisation

**Task:** Convert unstructured customer complaint narratives into structured JSON ticket metadata  
`{ product, sub_product, issue, sub_issue }`

**Model:** `Qwen/Qwen2.5-1.5B-Instruct` + LoRA fine-tuning  
**Backend:** ROCm / HIP (AMD GPU) — no CUDA, no bitsandbytes  

---

## Notebook Structure
| # | Section |
|---|---------|
| 1 | Environment & dependency check |
| 2 | Config & constants |
| 3 | Directory setup |
| 4 | Model & tokenizer loading |
| 5 | Data loading & sampling |
| 6 | Dataset formatting & tokenisation |
| 7 | Evaluation utilities (generative + structured) |
| 8 | **Qualitative inference demo — baseline** |
| 9 | Baseline quantitative evaluation |
| 10 | LoRA setup & training |
| 11 | **Qualitative inference demo — fine-tuned** |
| 12 | Post-training quantitative evaluation |
| 13 | Results comparison |
| 14 | Save adapter |


## 1. Environment & Dependency Check

In [ ]:
# ── Install required packages (run once) ─────────────────────────────────────
# Uncomment if running for the first time.

# !pip install -q transformers==4.44.0 peft==0.12.0 accelerate==0.34.0 \
#              datasets==2.21.0 trl==0.10.1 \
#              rouge-score sacrebleu nltk


In [ ]:
import torch

# ── ROCm / HIP device check ───────────────────────────────────────────────────
# On ROCm, torch.cuda.* APIs map to AMD GPUs via HIP.
print(f"PyTorch version    : {torch.__version__}")
print(f"ROCm/HIP available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU device         : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will run on CPU (very slow).")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device       : {DEVICE}")


## 2. Config & Constants

All tunable parameters live here — nothing else in the notebook needs to change.


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR         = "/workspace/shared"
MODEL_SAVE_DIR   = f"{BASE_DIR}/models/qwen2.5-1.5b"
ADAPTER_SAVE_DIR = f"{BASE_DIR}/models/qwen2.5-1.5b-lora"
CHECKPOINT_DIR   = f"{BASE_DIR}/checkpoints"
RESULTS_DIR      = f"{BASE_DIR}/results"

TRAIN_PATH = f"{BASE_DIR}/CFPB-Dataset-for-qwen/train.jsonl"
VAL_PATH   = f"{BASE_DIR}/CFPB-Dataset-for-qwen/validation.jsonl"
TEST_PATH  = f"{BASE_DIR}/CFPB-Dataset-for-qwen/test.jsonl"

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# ── Dataset sizes ─────────────────────────────────────────────────────────────
N_TRAIN = 500
N_VAL   = 250
N_TEST  = 250
RANDOM_SEED = 42

# ── Tokenisation ──────────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 1024

# ── LoRA hyperparameters ──────────────────────────────────────────────────────
LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# ── Training hyperparameters ──────────────────────────────────────────────────
NUM_EPOCHS           = 3
BATCH_SIZE           = 2
GRADIENT_ACCUM_STEPS = 8     # effective batch = 2 * 8 = 16
LEARNING_RATE        = 2e-4
LOGGING_STEPS        = 10
EVAL_STEPS           = 25
SAVE_STEPS           = 25

# ── Inference ─────────────────────────────────────────────────────────────────
MAX_NEW_TOKENS = 128    # enough to produce the JSON output

# ── Qualitative demo — fixed examples shown before & after training ────────────
# Indices into test_data. Set to None to pick randomly.
DEMO_INDICES = [0, 1, 2, 3, 4]   # 5 examples shown side-by-side


## 3. Directory Setup

In [ ]:
import os

def setup_directories(*dirs: str) -> None:
    for d in dirs:
        os.makedirs(d, exist_ok=True)
        print(f"  Ready: {d}")

print("Setting up directories...")
setup_directories(MODEL_SAVE_DIR, ADAPTER_SAVE_DIR, CHECKPOINT_DIR, RESULTS_DIR)


## 4. Model & Tokenizer Loading

Loaded in **bfloat16** — the native precision for AMD MI-series GPUs on ROCm.  
`bitsandbytes` (4-bit quantisation) is CUDA-only and is intentionally omitted.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model_and_tokenizer(model_name: str):
    """
    Load Qwen model and tokenizer.

    ROCm note: bfloat16 over float16 — better numerical stability on AMD GPUs.
    bitsandbytes 4-bit quantisation is CUDA-only; omitted here deliberately.
    """
    print(f"Loading tokenizer : {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Qwen models may ship without a pad token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Loading model     : {model_name}")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,   # optimal on ROCm / AMD GPUs
        device_map="auto",
    )

    print(f"Parameters        : {model.num_parameters():,}")
    return model, tokenizer


model, tokenizer = load_model_and_tokenizer(MODEL_NAME)


## 5. Data Loading & Sampling

Each JSONL line contains a `"messages"` key — a list of chat turns  
`[{"role": "system"|"user"|"assistant", "content": "..."}]`.

The **assistant turn** is the structured JSON output the model must learn to produce.


In [ ]:
import json
import random
from typing import List, Dict, Any

def load_jsonl(path: str) -> List[Dict[str, Any]]:
    """Read a JSONL file — one JSON object per line."""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data


def sample_data(data: List, n: int, seed: int = RANDOM_SEED) -> List:
    """Reproducibly sample up to n examples."""
    random.seed(seed)
    return random.sample(data, min(n, len(data)))


print("Loading dataset splits...")
raw_train = load_jsonl(TRAIN_PATH)
raw_val   = load_jsonl(VAL_PATH)
raw_test  = load_jsonl(TEST_PATH)

print(f"  Full train : {len(raw_train):,}")
print(f"  Full val   : {len(raw_val):,}")
print(f"  Full test  : {len(raw_test):,}")

train_data = sample_data(raw_train, N_TRAIN)
val_data   = sample_data(raw_val,   N_VAL)
test_data  = sample_data(raw_test,  N_TEST)

print(f"\nSampled — train: {len(train_data)} | val: {len(val_data)} | test: {len(test_data)}")

# ── Sanity check: print one full example ─────────────────────────────────────
print("\nSample example (messages):")
for turn in train_data[0]["messages"]:
    role    = turn["role"].upper()
    preview = turn["content"][:200].replace("\n", " ")
    print(f"  [{role}] {preview}")


## 6. Dataset Formatting & Tokenisation

We apply the Qwen **chat template** (which inserts `<|im_start|>` / `<|im_end|>` tokens)  
then tokenise. All non-tensor columns (`messages`, `text`) are dropped so the  
DataCollator only sees `input_ids` and `attention_mask`.


In [ ]:
from datasets import Dataset

def format_chat(example: Dict[str, Any]) -> Dict[str, str]:
    """Apply Qwen chat template — converts message list to a single training string."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,   # assistant turn already present
    )
    return {"text": text}


def tokenize_example(example: Dict[str, Any]) -> Dict:
    """
    Tokenise the formatted string.
    padding=False — padding happens per-batch in the DataCollator (more efficient).
    """
    return tokenizer(
        example["text"],
        truncation=True,
        padding=False,
        max_length=MAX_SEQ_LENGTH,
    )


def build_hf_dataset(data: List[Dict]) -> Dataset:
    """
    Build a HuggingFace Dataset ready for the Trainer.

    The column-drop step is critical: the raw 'messages' column (list of dicts)
    and 'text' column (string) cause a ValueError when the DataCollator tries
    to batch them into tensors. Keeping only tokenized columns fixes this.
    """
    ds = Dataset.from_list(data)
    ds = ds.map(format_chat)
    ds = ds.map(tokenize_example, batched=True)

    keep = {"input_ids", "attention_mask", "labels"}
    ds   = ds.remove_columns([c for c in ds.column_names if c not in keep])
    return ds


print("Formatting and tokenising datasets...")
train_ds = build_hf_dataset(train_data)
val_ds   = build_hf_dataset(val_data)

print(f"  Train : {len(train_ds)} examples | columns: {train_ds.column_names}")
print(f"  Val   : {len(val_ds)} examples")
print(f"\nFirst example — first 20 token ids: {train_ds[0]['input_ids'][:20]}")


## 7. Evaluation Utilities

### Two complementary evaluation layers

| Layer | Metrics | Why it matters |
|-------|---------|----------------|
| **Structured / Field-level** | Exact Match per field, Field Accuracy, Exact JSON Match, Micro/Macro/Weighted F1 | The model's job is to produce correct ticket metadata — field accuracy directly measures business value |
| **Generative** | ROUGE-1/2/L, BLEU, SacreBLEU, METEOR | Measures output fluency and n-gram overlap; useful for regression and catching degradation |

> **Why both?** A prediction of `"Checking account"` vs `"Checking or savings account"` scores ~0 on BLEU  
> but may be partially correct at the field level. Using only generative metrics on a structured extraction  
> task gives a misleading picture.


In [ ]:
import re
import json as _json
from rouge_score import rouge_scorer as _rouge
import sacrebleu as _sacrebleu
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from sklearn.metrics import f1_score
from tqdm import tqdm

# ── The four JSON fields this model must predict ───────────────────────────────
JSON_FIELDS = ["product", "sub_product", "issue", "sub_issue"]


# ─────────────────────────────────────────────────────────────────────────────
# Inference helpers
# ─────────────────────────────────────────────────────────────────────────────

def build_inference_prompt(messages: List[Dict]) -> str:
    """
    Strip the final assistant turn so the model must generate it.
    add_generation_prompt=True inserts the assistant-turn opening token.
    """
    return tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True,
    )


def predict(messages: List[Dict], model) -> str:
    """
    Greedy decode a single example.
    Returns only the newly generated tokens (the model's answer).
    """
    prompt = build_inference_prompt(messages)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    model.eval()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True)


def extract_reference(messages: List[Dict]) -> str:
    """Ground-truth: content of the last (assistant) message."""
    return messages[-1]["content"]


# ─────────────────────────────────────────────────────────────────────────────
# JSON parsing
# ─────────────────────────────────────────────────────────────────────────────

def parse_json_output(text: str) -> Dict[str, str]:
    """
    Attempt to parse a JSON dict from the model's output.

    The model may wrap the JSON in markdown code fences or produce minor
    formatting noise — we strip those and fall back gracefully.

    Returns a dict (possibly empty if parsing fails entirely).
    """
    # Strip markdown code fences if present
    text = re.sub(r"```(?:json)?", "", text).strip().rstrip("`").strip()

    # Try direct parse first
    try:
        parsed = _json.loads(text)
        if isinstance(parsed, dict):
            return {k: str(v).strip().lower() for k, v in parsed.items()}
    except _json.JSONDecodeError:
        pass

    # Fallback: extract the first {...} block via regex
    match = re.search(r"\{[^{}]+\}", text, re.DOTALL)
    if match:
        try:
            parsed = _json.loads(match.group())
            if isinstance(parsed, dict):
                return {k: str(v).strip().lower() for k, v in parsed.items()}
        except _json.JSONDecodeError:
            pass

    return {}   # parsing failed — treat as all-wrong for metric purposes


# ─────────────────────────────────────────────────────────────────────────────
# Structured / field-level metrics
# ─────────────────────────────────────────────────────────────────────────────

def compute_structured_metrics(
    predictions: List[str],
    references: List[str],
) -> Dict[str, Any]:
    """
    Field-level evaluation for the structured JSON output.

    Metrics computed
    ----------------
    exact_json_match  : fraction of examples where the full parsed dict matches
    field_accuracy    : per-field and averaged exact-match accuracy
    micro_f1          : sklearn micro-averaged F1 across all fields combined
    macro_f1          : sklearn macro-averaged F1
    weighted_f1       : sklearn weighted F1
    """
    parsed_preds = [parse_json_output(p) for p in predictions]
    parsed_refs  = [parse_json_output(r) for r in references]

    # ── Exact JSON match ──────────────────────────────────────────────────────
    exact_json = sum(
        p == r for p, r in zip(parsed_preds, parsed_refs)
    ) / len(predictions)

    # ── Per-field accuracy ────────────────────────────────────────────────────
    field_acc = {}
    for field in JSON_FIELDS:
        correct = sum(
            p.get(field, "__missing__") == r.get(field, "__missing__")
            for p, r in zip(parsed_preds, parsed_refs)
        )
        field_acc[field] = round(correct / len(predictions), 4)

    avg_field_acc = round(sum(field_acc.values()) / len(JSON_FIELDS), 4)

    # ── F1 scores (flatten all fields into one classification problem) ────────
    # Each example contributes one label per field.
    flat_preds, flat_refs = [], []
    for p_dict, r_dict in zip(parsed_preds, parsed_refs):
        for field in JSON_FIELDS:
            flat_preds.append(p_dict.get(field, "__missing__"))
            flat_refs.append(r_dict.get(field, "__missing__"))

    micro_f1    = round(f1_score(flat_refs, flat_preds, average="micro",    zero_division=0), 4)
    macro_f1    = round(f1_score(flat_refs, flat_preds, average="macro",    zero_division=0), 4)
    weighted_f1 = round(f1_score(flat_refs, flat_preds, average="weighted", zero_division=0), 4)

    return {
        "exact_json_match" : round(exact_json, 4),
        "avg_field_accuracy": avg_field_acc,
        "field_accuracy"   : field_acc,
        "micro_f1"         : micro_f1,
        "macro_f1"         : macro_f1,
        "weighted_f1"      : weighted_f1,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Generative metrics (ROUGE, BLEU, SacreBLEU, METEOR)
# ─────────────────────────────────────────────────────────────────────────────

def compute_generative_metrics(
    predictions: List[str],
    references: List[str],
) -> Dict[str, float]:
    """
    Standard NLG metrics — useful for regression tracking and completeness.
    Not the primary signal for this task (use structured metrics for that).
    """
    import nltk
    nltk.download("wordnet", quiet=True)
    from nltk.translate.meteor_score import meteor_score

    scorer = _rouge.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    r1, r2, rL, meteor_scores = [], [], [], []
    for pred, ref in zip(predictions, references):
        s = scorer.score(ref, pred)
        r1.append(s["rouge1"].fmeasure)
        r2.append(s["rouge2"].fmeasure)
        rL.append(s["rougeL"].fmeasure)
        # METEOR expects tokenised lists
        meteor_scores.append(
            meteor_score([ref.split()], pred.split())
        )

    ref_tok  = [[r.split()] for r in references]
    pred_tok = [p.split()   for p in predictions]
    bleu = corpus_bleu(ref_tok, pred_tok, smoothing_function=SmoothingFunction().method1)

    sacre = _sacrebleu.corpus_bleu(predictions, [references])

    avg = lambda lst: round(sum(lst) / len(lst), 4)
    return {
        "rouge1"   : avg(r1),
        "rouge2"   : avg(r2),
        "rougeL"   : avg(rL),
        "bleu"     : round(bleu, 4),
        "sacrebleu": round(sacre.score, 2),   # 0-100 scale
        "meteor"   : avg(meteor_scores),
    }


# ─────────────────────────────────────────────────────────────────────────────
# Full evaluation pipeline
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_model(
    model,
    data: List[Dict],
    label: str = "Evaluation",
) -> Dict[str, Any]:
    """
    Run inference on every example in `data`, then compute both
    structured and generative metrics.

    Returns a dict with keys: 'structured' and 'generative'.
    """
    predictions, references = [], []

    for example in tqdm(data, desc=label):
        pred = predict(example["messages"], model)
        ref  = extract_reference(example["messages"])
        predictions.append(pred)
        references.append(ref)

    return {
        "structured" : compute_structured_metrics(predictions, references),
        "generative" : compute_generative_metrics(predictions, references),
        # Store raw outputs for the qualitative demo (re-used later)
        "_predictions": predictions,
        "_references" : references,
    }


def save_metrics(metrics: Dict, path: str) -> None:
    """Persist metrics dict to JSON (strip internal _ keys)."""
    clean = {k: v for k, v in metrics.items() if not k.startswith("_")}
    with open(path, "w") as f:
        _json.dump(clean, f, indent=2)
    print(f"Saved: {path}")


## 8. Qualitative Inference Demo — Baseline Model

Before running numbers, see exactly what the **untrained** model produces  
for a handful of real complaints. This makes the quantitative improvement  
tangible and interpretable.


In [ ]:
def run_qualitative_demo(model, data: List[Dict], indices: List[int], label: str) -> None:
    """
    Print a side-by-side view of complaint → prediction vs reference
    for a small set of hand-picked examples.
    """
    print(f"\n{'═'*70}")
    print(f"  QUALITATIVE DEMO — {label}")
    print(f"{'═'*70}")

    for i, idx in enumerate(indices):
        example  = data[idx]
        messages = example["messages"]

        # Extract the user complaint (typically the last user turn)
        user_content = next(
            (m["content"] for m in reversed(messages) if m["role"] == "user"), ""
        )
        reference = extract_reference(messages)
        prediction = predict(messages, model)

        # Parse both to check field alignment
        pred_parsed = parse_json_output(prediction)
        ref_parsed  = parse_json_output(reference)

        print(f"\n{'─'*70}")
        print(f"  Example {i+1} (index {idx})")
        print(f"{'─'*70}")

        # Show a trimmed version of the complaint
        complaint_preview = user_content[:400].replace("\n", " ").strip()
        if len(user_content) > 400:
            complaint_preview += "..."
        print(f"\n  COMPLAINT:\n  {complaint_preview}")

        print(f"\n  REFERENCE (ground truth):")
        print(f"  {reference.strip()}")

        print(f"\n  MODEL PREDICTION:")
        print(f"  {prediction.strip()}")

        # Field-level match summary
        if ref_parsed:
            print(f"\n  FIELD MATCH:")
            for field in JSON_FIELDS:
                ref_val  = ref_parsed.get(field, "—")
                pred_val = pred_parsed.get(field, "MISSING")
                match    = "✓" if ref_val == pred_val else "✗"
                print(f"    {match} {field:<14} ref='{ref_val}'   pred='{pred_val}'")

    print(f"\n{'═'*70}\n")


# Run the baseline demo before any training
demo_indices = DEMO_INDICES if DEMO_INDICES else list(range(5))
run_qualitative_demo(model, test_data, demo_indices, label="BASELINE (pre-training)")


## 9. Baseline Quantitative Evaluation

Full test-set evaluation on the untrained base model.  
These scores are the benchmark — every metric must improve after fine-tuning.


In [ ]:
print("Running baseline evaluation on full test set...")
baseline_results = evaluate_model(model, test_data, label="Baseline")

print("\n── Structured Metrics (baseline) ──────────────────────────────────────")
s = baseline_results["structured"]
print(f"  Exact JSON Match    : {s['exact_json_match']:.4f}")
print(f"  Avg Field Accuracy  : {s['avg_field_accuracy']:.4f}")
for field, acc in s["field_accuracy"].items():
    print(f"    {field:<16}: {acc:.4f}")
print(f"  Micro F1            : {s['micro_f1']:.4f}")
print(f"  Macro F1            : {s['macro_f1']:.4f}")
print(f"  Weighted F1         : {s['weighted_f1']:.4f}")

print("\n── Generative Metrics (baseline) ──────────────────────────────────────")
g = baseline_results["generative"]
for k, v in g.items():
    print(f"  {k:<12}: {v}")

save_metrics(baseline_results, f"{RESULTS_DIR}/baseline_metrics.json")


## 10. LoRA Setup & Training

**LoRA (Low-Rank Adaptation):** inserts small trainable matrices (rank `r`) into  
selected attention layers, freezing the base model. Only ~1% of parameters are updated,  
making this VRAM-efficient and fast on ROCm.

### ROCm-specific choices
- `bf16=True` — native AMD GPU precision (CDNA architecture)  
- `fp16=False` — numerically unstable on some AMD cards  
- `optim=adamw_torch` — replaces `paged_adamw_8bit` (bitsandbytes / CUDA-only)  
- `DataCollatorForLanguageModeling(mlm=False)` — correct collator for decoder-only CLM


In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

def setup_lora(model, r: int, alpha: int, dropout: float, targets: List[str]):
    """
    Attach LoRA adapters to the model.

    Only the adapter matrices (A, B) are trainable — base weights stay frozen.
    This reduces trainable parameters from ~1.5B to ~10-20M.
    """
    config = LoraConfig(
        r              = r,
        lora_alpha     = alpha,
        lora_dropout   = dropout,
        bias           = "none",
        task_type      = "CAUSAL_LM",
        target_modules = targets,
    )
    peft_model = get_peft_model(model, config)
    peft_model.print_trainable_parameters()
    return peft_model


def build_training_args(output_dir: str) -> TrainingArguments:
    """
    TrainingArguments tuned for ROCm / AMD GPUs.

    bf16=True, fp16=False  — AMD MI-series native precision
    optim=adamw_torch      — no bitsandbytes dependency
    dataloader_num_workers — 0 is the safe default on ROCm
    """
    return TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUM_STEPS,
        learning_rate               = LEARNING_RATE,
        bf16                        = True,
        fp16                        = False,
        optim                       = "adamw_torch",
        logging_steps               = LOGGING_STEPS,
        eval_steps                  = EVAL_STEPS,
        save_steps                  = SAVE_STEPS,
        eval_strategy               = "steps",
        save_strategy               = "steps",
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        report_to                   = "none",
        dataloader_num_workers      = 0,
        remove_unused_columns       = False,
    )


# ── Attach LoRA ────────────────────────────────────────────────────────────────
model = setup_lora(
    model,
    r       = LORA_R,
    alpha   = LORA_ALPHA,
    dropout = LORA_DROPOUT,
    targets = LORA_TARGET_MODULES,
)

# ── Collator ──────────────────────────────────────────────────────────────────
# DataCollatorForLanguageModeling is the correct choice for decoder-only CLM:
#   - Pads each batch to the batch-maximum length (not the global max)
#   - Sets labels = input_ids (standard CLM objective — predict next token)
#   - mlm=False disables masked-language-model masking
# pad_to_multiple_of=8 aligns tensor shapes for faster ROCm matrix multiplications.
data_collator = DataCollatorForLanguageModeling(
    tokenizer          = tokenizer,
    mlm                = False,
    pad_to_multiple_of = 8,
)

# ── Trainer ───────────────────────────────────────────────────────────────────
# `tokenizer` is NOT passed here — removed from Trainer.__init__ in
# transformers >= 4.46. All padding is handled by the DataCollator above.
training_args = build_training_args(CHECKPOINT_DIR)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    data_collator = data_collator,
)

print("Trainer initialised. Starting training...")
trainer.train()
print("Training complete.")


## 11. Qualitative Inference Demo — Fine-Tuned Model

Same examples, same format. Compare directly against Section 8 to see  
whether field predictions improved, and by how much.


In [ ]:
# Use the exact same indices so the comparison is apples-to-apples
run_qualitative_demo(model, test_data, demo_indices, label="FINE-TUNED (post-training)")


## 12. Post-Training Quantitative Evaluation

In [ ]:
print("Running post-training evaluation on full test set...")
final_results = evaluate_model(model, test_data, label="Fine-tuned")

print("\n── Structured Metrics (fine-tuned) ─────────────────────────────────────")
s = final_results["structured"]
print(f"  Exact JSON Match    : {s['exact_json_match']:.4f}")
print(f"  Avg Field Accuracy  : {s['avg_field_accuracy']:.4f}")
for field, acc in s["field_accuracy"].items():
    print(f"    {field:<16}: {acc:.4f}")
print(f"  Micro F1            : {s['micro_f1']:.4f}")
print(f"  Macro F1            : {s['macro_f1']:.4f}")
print(f"  Weighted F1         : {s['weighted_f1']:.4f}")

print("\n── Generative Metrics (fine-tuned) ─────────────────────────────────────")
g = final_results["generative"]
for k, v in g.items():
    print(f"  {k:<12}: {v}")

save_metrics(final_results, f"{RESULTS_DIR}/final_metrics.json")


## 13. Results Comparison — Baseline vs Fine-Tuned

In [ ]:
def print_comparison(baseline: Dict, final: Dict) -> None:
    """Print a formatted side-by-side metric comparison with delta."""

    def delta_str(b, f):
        d    = f - b
        sign = "+" if d >= 0 else ""
        return f"{sign}{d:.4f}"

    print("\n" + "═"*70)
    print("  STRUCTURED METRICS")
    print("═"*70)
    print(f"  {'Metric':<24}  {'Baseline':>10}  {'Fine-tuned':>12}  {'Δ Delta':>10}")
    print("  " + "─"*60)

    bs = baseline["structured"]
    fs = final["structured"]

    flat_s = {
        "Exact JSON Match"    : (bs["exact_json_match"],  fs["exact_json_match"]),
        "Avg Field Accuracy"  : (bs["avg_field_accuracy"], fs["avg_field_accuracy"]),
        "Micro F1"            : (bs["micro_f1"],           fs["micro_f1"]),
        "Macro F1"            : (bs["macro_f1"],           fs["macro_f1"]),
        "Weighted F1"         : (bs["weighted_f1"],        fs["weighted_f1"]),
    }
    for k, (b, f) in flat_s.items():
        print(f"  {k:<24}  {b:>10.4f}  {f:>12.4f}  {delta_str(b,f):>10}")

    print("\n  Field Accuracy (per field):")
    for field in JSON_FIELDS:
        b = bs["field_accuracy"][field]
        f = fs["field_accuracy"][field]
        print(f"    {field:<20}  {b:>10.4f}  {f:>12.4f}  {delta_str(b,f):>10}")

    print("\n" + "═"*70)
    print("  GENERATIVE METRICS")
    print("═"*70)
    print(f"  {'Metric':<24}  {'Baseline':>10}  {'Fine-tuned':>12}  {'Δ Delta':>10}")
    print("  " + "─"*60)

    bg = baseline["generative"]
    fg = final["generative"]

    for k in bg:
        b, f = bg[k], fg[k]
        print(f"  {k:<24}  {b:>10.4f}  {f:>12.4f}  {delta_str(b,f):>10}")

    print("═"*70 + "\n")


print_comparison(baseline_results, final_results)

# Save combined comparison
save_metrics(
    {"baseline": baseline_results, "fine_tuned": final_results},
    f"{RESULTS_DIR}/metrics_comparison.json",
)


## 14. Save LoRA Adapter

Only the adapter weights are saved (~10-50 MB vs several GB for the full model).  
To reload for inference, use `PeftModel.from_pretrained(base_model, adapter_path)`.


In [ ]:
def save_adapter(model, tokenizer, save_path: str) -> None:
    """Save the LoRA adapter weights and tokenizer."""
    print(f"Saving adapter to {save_path} ...")
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print("Saved.")


save_adapter(model, tokenizer, ADAPTER_SAVE_DIR)


In [ ]:
# ── Reload snippet (for future inference) ─────────────────────────────────────
# from peft import PeftModel
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch
#
# base  = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")
# tok   = AutoTokenizer.from_pretrained(ADAPTER_SAVE_DIR)
# model = PeftModel.from_pretrained(base, ADAPTER_SAVE_DIR)
# model.eval()
#
# Then call: predict(messages, model)

print("All done.")
print(f"  Adapter  → {ADAPTER_SAVE_DIR}")
print(f"  Results  → {RESULTS_DIR}")
